# 11_gold_validation
Automated checks on the Gold layer. Run it after `10_silver_to_gold` (or after a full pipeline run).
It checks table existence, grain, foreign keys, reconciliation with Silver, business rules, idempotency (it re-runs Gold for the latest date and expects 0 inserted / 0 updated) and logging.
Parameters: date (blank = latest date in Silver).

In [ ]:
dbutils.widgets.text("date", "")

In [ ]:
from pyspark.sql import functions as F

CATALOG, SCHEMA = "workspace", "steam"
spark.sql(f"USE {CATALOG}.{SCHEMA}")

test_date = dbutils.widgets.get("date").strip() or str(spark.sql("SELECT MAX(last_load_date) FROM silver_steam").first()[0])
test_key = int(test_date.replace("-", ""))
print("testing date:", test_date, "| date_key:", test_key)

results = []
def check(name, passed, detail=""):
    results.append((name, bool(passed)))
    print(("PASS" if passed else "FAIL"), "-", name, ("| " + str(detail)) if detail != "" else "")

def n(sql):
    return spark.sql(sql).first()[0]

In [ ]:
# 1. All Gold tables and views exist
for t in ["dim_date", "dim_price_tier", "dim_publisher", "dim_game", "fact_game_daily",
          "vw_fact_latest", "gold_publisher_summary", "gold_top_games", "gold_price_tier_analysis"]:
    check(f"{t} exists", spark.catalog.tableExists(t))

# 2. Grain: no duplicate (game_key, date_key); dimension keys are unique
check("fact grain unique (game_key, date_key)",
      n("SELECT COUNT(*) - COUNT(DISTINCT game_key, date_key) FROM fact_game_daily") == 0)
check("dim_game key unique", n("SELECT COUNT(*) - COUNT(DISTINCT game_key) FROM dim_game") == 0)
check("dim_publisher key unique", n("SELECT COUNT(*) - COUNT(DISTINCT publisher_key) FROM dim_publisher") == 0)
check("dim_date key unique", n("SELECT COUNT(*) - COUNT(DISTINCT date_key) FROM dim_date") == 0)

# 3. Foreign keys: every fact key exists in its dimension (0 orphans)
for dim, col in [("dim_game", "game_key"), ("dim_publisher", "publisher_key"),
                 ("dim_date", "date_key"), ("dim_price_tier", "price_tier_key")]:
    orphans = n(f"SELECT COUNT(*) FROM fact_game_daily f LEFT ANTI JOIN {dim} d ON f.{col} = d.{col}")
    check(f"no orphan {col} -> {dim}", orphans == 0, f"orphans={orphans}")

In [ ]:
# 4. Reconciliation with Silver for the tested date
silver_rows = n(f"SELECT COUNT(*) FROM silver_steam WHERE last_load_date = DATE'{test_date}'")
fact_rows = n(f"SELECT COUNT(*) FROM fact_game_daily WHERE date_key = {test_key}")
check("fact rows = silver rows changed on that date", silver_rows == fact_rows, f"silver={silver_rows} fact={fact_rows}")

s = spark.sql(f"SELECT COALESCE(SUM(ccu),0) c, COALESCE(SUM(total_reviews),0) r FROM silver_steam WHERE last_load_date = DATE'{test_date}'").first()
f = spark.sql(f"SELECT COALESCE(SUM(ccu),0) c, COALESCE(SUM(total_reviews),0) r FROM fact_game_daily WHERE date_key = {test_key}").first()
check("SUM(ccu) matches Silver", s.c == f.c, f"silver={s.c} fact={f.c}")
check("SUM(total_reviews) matches Silver", s.r == f.r, f"silver={s.r} fact={f.r}")

# a date can never have more fact rows than Bronze rows loaded for it
over = n("""
    SELECT COUNT(*) FROM (
      SELECT f.date_key, COUNT(*) AS fact_rows FROM fact_game_daily f GROUP BY f.date_key) x
    LEFT JOIN (SELECT CAST(date_format(load_date, 'yyyyMMdd') AS INT) AS date_key, COUNT(*) AS bronze_rows
               FROM bronze_steam GROUP BY 1) b ON x.date_key = b.date_key
    WHERE b.bronze_rows IS NULL OR x.fact_rows > b.bronze_rows
""")
check("fact rows per date <= bronze rows per date", over == 0, f"violations={over}")

In [ ]:
# 5. Business rules
check("approval_rate between 0 and 1",
      n("SELECT COUNT(*) FROM fact_game_daily WHERE approval_rate < 0 OR approval_rate > 1") == 0)
check("owners_min <= owners_max",
      n("SELECT COUNT(*) FROM fact_game_daily WHERE owners_min > owners_max") == 0)
check("no negative price or ccu",
      n("SELECT COUNT(*) FROM fact_game_daily WHERE price_usd < 0 OR ccu < 0") == 0)
check("Free tier <=> price = 0",
      n("SELECT COUNT(*) FROM fact_game_daily WHERE (price_tier_key = 1) <> (price_usd = 0)") == 0)
check("price tier boundaries correct",
      n("""SELECT COUNT(*) FROM fact_game_daily WHERE
              (price_usd > 0 AND price_usd < 5 AND price_tier_key <> 2)
           OR (price_usd >= 5 AND price_usd <= 20 AND price_tier_key <> 3)
           OR (price_usd > 20 AND price_tier_key <> 4)""") == 0)

In [ ]:
# 6. Idempotency: re-run Gold for the same date, nothing may change
before = spark.sql("SELECT (SELECT COUNT(*) FROM fact_game_daily) f, (SELECT COUNT(*) FROM dim_game) g, (SELECT COUNT(*) FROM dim_publisher) p").first()
dbutils.notebook.run("./10_silver_to_gold", 3600, {"date": test_date})
after = spark.sql("SELECT (SELECT COUNT(*) FROM fact_game_daily) f, (SELECT COUNT(*) FROM dim_game) g, (SELECT COUNT(*) FROM dim_publisher) p").first()
check("re-run leaves row counts unchanged", tuple(before) == tuple(after), f"before={tuple(before)} after={tuple(after)}")

last3 = spark.sql("""
    SELECT file_processed, status, rows_inserted, rows_updated, start_time, end_time
    FROM pipeline_execution_logs WHERE layer = 'Silver-to-Gold'
    ORDER BY start_time DESC LIMIT 3""").collect()
check("re-run logged 0 inserted / 0 updated on all 3 tables",
      len(last3) == 3 and all(r.rows_inserted == 0 and r.rows_updated == 0 for r in last3),
      [(r.file_processed, r.rows_inserted, r.rows_updated) for r in last3])

# 7. Logging
check("Silver-to-Gold log rows exist and succeeded",
      len(last3) == 3 and all(r.status == "Success" for r in last3))
check("log end_time >= start_time", all(r.end_time >= r.start_time for r in last3))

In [ ]:
passed = sum(1 for _, ok in results if ok)
print(f"\n{passed} / {len(results)} checks passed")
failed = [name for name, ok in results if not ok]
if failed:
    raise AssertionError("Failed checks: " + "; ".join(failed))
print("Gold layer is good to go.")

## Performance demo for the report (optional)
Shows the fact table layout before and after `OPTIMIZE`. Liquid Clustering on `(date_key, game_key)` was set when the table was created, and `OPTIMIZE` applies it to the data files. Take a screenshot of the two outputs for the FinOps / optimization section of the report.

In [ ]:
display(spark.sql("DESCRIBE DETAIL fact_game_daily").select("numFiles", "sizeInBytes", "clusteringColumns"))
spark.sql("OPTIMIZE fact_game_daily")
display(spark.sql("DESCRIBE DETAIL fact_game_daily").select("numFiles", "sizeInBytes", "clusteringColumns"))